In [104]:
import pandas as pd


In [105]:
# medicaid_path = "../../workspaces/MEDICAID_AGGREGATE20.CSV"
medicaid_path = '/home/daniel.lien/dev/homework/Chronic_Disease_Indicators_Data_Visualization/workspaces/ashley/MEDICAID_AGGREGATE20.CSV'

In [106]:
df = pd.read_csv(medicaid_path)
df.head()

,Code,Item,Group,Region_Number,Region_Name,State_Name,Y1991,Y1992,Y1993,Y1994,...,Y2012,Y2013,Y2014,Y2015,Y2016,Y2017,Y2018,Y2019,Y2020,Average_Annual_Percent_Growth
0,1,Medicaid/Personal Health Care (Millions of Dol...,United States,0,United States,NaN,88921,103417,116453,126856,...,388256,405666,446921,484506,503382,516067,531782,552953,586914,6.7
1,1,Medicaid/Personal Health Care (Millions of Dol...,Region,1,New England,NaN,6676,7418,7768,8896,...,25512,26113,28519,30368,31616,31577,32910,32767,34183,5.8
2,1,Medicaid/Personal Health Care (Millions of Dol...,Region,2,Mideast,NaN,24821,28301,30979,33776,...,87340,88864,97912,103253,109981,117574,122226,123410,125408,5.7
3,1,Medicaid/Personal Health Care (Millions of Dol...,Region,3,Great Lakes,NaN,13820,16525,18627,19492,...,52941,56964,62039,66741,68168,70092,70563,74817,83497,6.4
4,1,Medicaid/Personal Health Care (Millions of Dol...,Region,4,Plains,NaN,5548,6569,7205,7910,...,25535,26266,27973,29466,30255,31077,32582,33444,34551,6.5


In [107]:
# Medicaid Cleaning
"""
Drop NA
Filter by  'region == state'
Filter out non-states
group by state and year and sum all expenses  
filter out years not >2011

Pivot to get by state and year 
"""

"\nDrop NA\nFilter by  'region == state'\nFilter out non-states\ngroup by state and year and sum all expenses  \nfilter out years not >2011\n\nPivot to get by state and year \n"

In [108]:
cols_to_drop = [
    'Code',
    'Region_Number',
    'Region_Name',
    'Y1991', 'Y1992', 'Y1993', 'Y1994','Y1995','Y1996','Y1997','Y1998','Y1999', 'Y2000', 'Y2001','Y2002','Y2003','Y2004','Y2005','Y2006','Y2007','Y2008','Y2009','Y2010',
    'Average_Annual_Percent_Growth'
]
df = df.drop(columns=cols_to_drop)

In [109]:
# Filter by 'Region' == 'State Name'
df = df[df['Group'] == 'State']

In [110]:
df = df.dropna()

In [111]:
df = df[df['State_Name'] != 'District of Columbia']

In [112]:
year_columns = [col for col in df.columns if col.startswith("Y")]  # Identifying year columns
df[year_columns] = df[year_columns].apply(pd.to_numeric, errors='coerce')

In [113]:
# Melt the DataFrame to long format
df_long = df.melt(id_vars=['State_Name'], value_vars=year_columns, 
                   var_name='Year', value_name='Expenses')

# Convert 'Year' to integer
df_long['Year'] = df_long['Year'].str.extract('(\\d+)').astype(int)

# Group by state and year, summing Medicaid expenses
df_grouped = df_long.groupby(['State_Name', 'Year'])['Expenses'].sum().reset_index()

In [114]:
# Rename Columns to 
df_grouped.rename(columns={'YearStart': 'Year',
                    'State_Name': 'State',
                    'Expenses': 'Medicaid_Spending',
                    }, inplace=True)

In [115]:
output_dir = '/home/daniel.lien/dev/homework/data/medicaid_data_cleaned.csv'
df_grouped.to_csv(output_dir, index=False)